
# Movias — Fase 1: **Limpeza e Preparação de Dados** (Kilometragem & Horas)

Este notebook documenta **minuciosamente** o pipeline de limpeza e preparação para séries diárias por veículo,
considerando os objetivos de previsão de **manutenções por quilometragem** e **por horas de utilização**.

**Estratégia de outliers escolhida: **Híbrido** (regra fixa + método estatístico IQR/Z-score por veículo).

> **Entradas esperadas** (por dia e por veículo):  
> `veiculo_id, data, km_dia_por_soma, km_dia_por_odometro, odo_ini_dia, odo_fim_dia, atividade_total_h, viagens_qtd, flag_reset_odometro, fimdesemana`  
>
> **Saídas principais:**  
> - Série **diária** com `km_dia_final`, `h_dia_final`, cumulativos e *features* derivadas;  
> - Dados **consistentes** (monotonicidade de odômetro, tratamento de lacunas e outliers);  
> - Arquivos prontos para a Fase 2 (forecasting).


## 0. Setup do ambiente e bibliotecas

In [ ]:
import os, math, numpy as np, pandas as pd
from datetime import timedelta
from pathlib import Path
import shutil


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

#pip install pyarrow



## 1. Ingestão e inspeção inicial dos dados

- Leitura do arquivo   
- Em produção, leremos uma **tabela consolidada** (todos os veículos ) do PostgreSQL ou CSVs particionados.


In [ ]:

FILENAME = "resultado_movias_all.csv"

# ---------- Localização robusta do arquivo ----------
def find_data_raw(filename=FILENAME):
    cwd = Path.cwd().resolve()
    direct = cwd / "data" / "raw" / filename
    if direct.exists():
        return direct
    for base in [cwd, *cwd.parents]:
        candidate = base / "data" / "raw" / filename
        if candidate.exists():
            return candidate
    matches = list(cwd.glob(f"**/data/raw/{filename}"))
    if matches:
        matches.sort(key=lambda p: len(str(p)))
        return matches[0]
    return None

EXAMPLE_FILE = find_data_raw(FILENAME)
print("CWD:", Path.cwd())
print("EXAMPLE_FILE:", EXAMPLE_FILE)
assert EXAMPLE_FILE is not None, "Arquivo não encontrado: coloque-o em data/raw/resultado_movias_all.csv"

# ---------- Detecção simples de separador ----------
first_line = EXAMPLE_FILE.read_text(encoding="utf-8", errors="ignore").splitlines()[:1]
sep = "|" if ("|" in first_line[0]) else (";" if ";" in first_line[0] else ("," if "," in first_line[0] else ","))
print("Separador detectado:", sep)

# ---------- Leitura: tudo como string ----------
df_raw = pd.read_csv(
    EXAMPLE_FILE,
    sep=sep,
    dtype="string",
    low_memory=False,
    na_values=["", "NA", "NaN", None]
)

# ---------- Funções auxiliares ----------
def to_bool01(series):
    s = series.astype("string").str.strip().str.lower()
    true_set  = {"true","1","sim","s","yes","y","t"}
    false_set = {"false","0","nao","não","n","no","f"}
    out = pd.Series(pd.NA, index=series.index, dtype="Int64")
    out = out.mask(s.isin(true_set), 1)
    out = out.mask(s.isin(false_set), 0)
    return out

def parse_date_robusto(s: pd.Series) -> pd.Series:
    """Parse robusto tentando múltiplos formatos; mantém NaT nos irrecuperáveis."""
    s0 = s.astype("string").str.strip()

    # 1) tenta conversão automática — infer_datetime_format removido
    d = pd.to_datetime(s0, errors="coerce")

    # 2) tenta dd/mm/aaaa
    mask = d.isna()
    if mask.any():
        d.loc[mask] = pd.to_datetime(s0[mask], format="%d/%m/%Y", errors="coerce")

    # 3) tenta aaaa-mm-dd
    mask = d.isna()
    if mask.any():
        d.loc[mask] = pd.to_datetime(s0[mask], format="%Y-%m-%d", errors="coerce")

    # 4) tenta dd-mm-aaaa
    mask = d.isna()
    if mask.any():
        d.loc[mask] = pd.to_datetime(s0[mask], format="%d-%m-%Y", errors="coerce")

    # 5) tenta aaaa/mm/dd
    mask = d.isna()
    if mask.any():
        d.loc[mask] = pd.to_datetime(s0[mask], format="%Y/%m/%d", errors="coerce")

    return d
# ---------- Tipagem explícita (sem cliente_id) ----------

# 1) Datas — guardar original e converter de forma robusta
if "data" in df_raw.columns:
    df_raw["data_original_str"] = df_raw["data"].astype("string").str.strip()  # para diagnóstico
    df_raw["data"] = parse_date_robusto(df_raw["data"])
    df_raw["data_br"] = df_raw["data"].dt.strftime("%d/%m/%Y")

# 2) IDs como string
if "veiculo_id" in df_raw.columns:
    df_raw["veiculo_id"] = df_raw["veiculo_id"].astype("string")

# 3) Numéricos contínuos
for c in ["km_dia_por_soma", "km_dia_por_odometro", "odo_ini_dia", "odo_fim_dia", "atividade_total_h"]:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

# 4) Contagens/flags numéricos (Int64)
if "viagens_qtd" in df_raw.columns:
    df_raw["viagens_qtd"] = pd.to_numeric(df_raw["viagens_qtd"], errors="coerce").astype("Int64")

# 5) Flags booleanos textuais → 0/1
if "flag_reset_odometro" in df_raw.columns:
    tmp = to_bool01(df_raw["flag_reset_odometro"])
    mask_na = tmp.isna()
    if mask_na.any():
        num = pd.to_numeric(df_raw.loc[mask_na, "flag_reset_odometro"], errors="coerce")
        tmp.loc[mask_na & num.notna()] = num.loc[mask_na & num.notna()].astype("Int64")
    df_raw["flag_reset_odometro"] = tmp.fillna(0).astype("Int64")

if "fimdesemana" in df_raw.columns:
    fimde01 = to_bool01(df_raw["fimdesemana"])
    mask_na = fimde01.isna()
    if mask_na.any():
        num = pd.to_numeric(df_raw.loc[mask_na, "fimdesemana"], errors="coerce")
        fimde01.loc[mask_na & num.notna()] = num.loc[mask_na & num.notna()].astype("Int64")
    if "data" in df_raw.columns:
        mask_na = fimde01.isna()
        if mask_na.any():
            derived = df_raw["data"].dt.weekday.isin([5,6]).astype(int).astype("Int64")
            fimde01 = fimde01.mask(mask_na, derived)
    df_raw["fimdesemana"] = fimde01.fillna(0).astype("Int64")

print("Linhas lidas:", len(df_raw))

# ---------- Diagnóstico: quais veiculo_id ficaram com data nula ----------
if "data" in df_raw.columns:
    mask_nat = df_raw["data"].isna()
    if mask_nat.any():
        # 1) veiculos com NaT
        veiculos_com_nat = (df_raw.loc[mask_nat, "veiculo_id"]
                            .dropna().unique().tolist())
        print("veiculo_id com 'data' nula:", veiculos_com_nat)

        # 2) contagem por veiculo_id
        contagem_nat = (df_raw.loc[mask_nat]
                        .groupby("veiculo_id", dropna=True)["data"]
                        .size()
                        .sort_values(ascending=False))
        print("\nContagem de linhas com 'data' nula por veiculo_id:")
        print(contagem_nat.head(20))

        # 3) exemplos das strings problemáticas de data
        exemplos = (df_raw.loc[mask_nat, ["veiculo_id","data_original_str"]]
                    .drop_duplicates()
                    .head(20))
        print("\nExemplos de strings de 'data' que falharam no parse:")
        print(exemplos.to_string(index=False))
    else:
        print("Nenhum 'data' nulo após o parse.")

# Preview
cols_preview = [c for c in ["veiculo_id","data_br","viagens_qtd","km_dia_por_soma","km_dia_por_odometro",
                            "atividade_total_h","odo_ini_dia","odo_fim_dia","flag_reset_odometro","fimdesemana"]
                if c in df_raw.columns]
df_raw[cols_preview].head(5)


CWD: C:\Users\baj00\movias_forecast\notebooks
EXAMPLE_FILE: C:\Users\baj00\movias_forecast\data\raw\resultado_movias_all.csv
Separador detectado: ,
Linhas lidas: 22124296
Nenhum 'data' nulo após o parse.


,veiculo_id,data_br,viagens_qtd,km_dia_por_soma,km_dia_por_odometro,atividade_total_h,odo_ini_dia,odo_fim_dia,flag_reset_odometro,fimdesemana
0,1,01/01/2022,0,0.0,0.0,0.0,<NA>,<NA>,0,1
1,1,02/01/2022,0,0.0,0.0,0.0,<NA>,<NA>,0,1
2,1,03/01/2022,0,0.0,0.0,0.0,<NA>,<NA>,0,0
3,1,04/01/2022,0,0.0,0.0,0.0,<NA>,<NA>,0,0
4,1,05/01/2022,0,0.0,0.0,0.0,<NA>,<NA>,0,0


## 2. Validação de esquema mínimo
Garantimos a presença e os tipos dos campos essenciais.

In [ ]:
# ============================================================
# 2) Validação de esquema mínimo
# - Presença de colunas
# - Tipos esperados (tolerante a Float64)
# - Nulos em chaves
# - Nulos em demais colunas
# - Duplicatas em (veiculo_id, data)
# - Flags binárias (0/1)
# - Não-negatividade (aviso)
# Gera: schema_report (DataFrame)
# Requer: df_raw do Bloco 1
# ============================================================

# ----- Colunas essenciais (sem cliente_id) -----
REQUIRED_COLS = [
    "veiculo_id", "data",
    "km_dia_por_soma", "km_dia_por_odometro",
    "odo_ini_dia", "odo_fim_dia",
    "atividade_total_h", "viagens_qtd",
    "flag_reset_odometro", "fimdesemana"
]

# ----- Dtypes esperados após Bloco 1 -----
EXPECTED_DTYPES = {
    "veiculo_id":            "string",
    "data":                  "datetime64[ns]",
    "km_dia_por_soma":       "float",     # aceita Float64/float64
    "km_dia_por_odometro":   "float",
    "odo_ini_dia":           "float",
    "odo_fim_dia":           "float",
    "atividade_total_h":     "float",
    "viagens_qtd":           "Int64",
    "flag_reset_odometro":   "Int64",
    "fimdesemana":           "Int64",
}

# ----- Helpers -----
def _dtype_name(series: pd.Series) -> str:
    """
    Normaliza nomes de dtype p/ comparação com EXPECTED_DTYPES.
    Trata 'Float64'/'float64' como 'float'; 'Int64' (nullable) como 'Int64';
    datetime como 'datetime64[ns]'; string como 'string'.
    """
    dt = str(series.dtype)
    if dt in ("Float64","float64","Float32","float32"):
        return "float"
    if dt.startswith("Int64"):
        return "Int64"
    if dt in ("int64","int32"):
        return "int64"
    if dt.startswith("datetime64"):
        return "datetime64[ns]"
    if "string" in dt:
        return "string"
    return dt

def check_required_columns(df: pd.DataFrame, required: list) -> dict:
    missing = [c for c in required if c not in df.columns]
    return {"missing_required_cols": missing, "ok": len(missing) == 0}

def check_dtypes(df: pd.DataFrame, expected: dict) -> list:
    errs = []
    for col, exp in expected.items():
        if col not in df.columns:
            errs.append((col, "missing", exp, None))
            continue
        got = _dtype_name(df[col])
        if got != exp:
            errs.append((col, "wrong_dtype", exp, got))
    return errs

def check_nulls(df: pd.DataFrame, cols: list, threshold=0.0) -> list:
    issues = []
    na_rate = df[cols].isna().mean(numeric_only=False)
    for c, rate in na_rate.items():
        if rate > threshold:
            issues.append((c, "null_rate", float(rate)))
    return issues

def check_duplicates(df: pd.DataFrame) -> int:
    if ("veiculo_id" in df.columns) and ("data" in df.columns):
        return int(df.duplicated(subset=["veiculo_id","data"]).sum())
    return 0

def check_flags_binary(df: pd.DataFrame, cols=("flag_reset_odometro","fimdesemana")) -> list:
    issues = []
    for c in cols:
        if c not in df.columns:
            issues.append((c, "missing_flag_col"))
            continue
        uniq = set(df[c].dropna().unique().tolist())
        if not uniq.issubset({0,1}):
            issues.append((c, "non_binary_values", sorted(uniq)))
    return issues

def check_nonnegative(df: pd.DataFrame, cols: list) -> list:
    issues = []
    for c in cols:
        if c in df.columns:
            has_neg = (df[c] < 0).any()
            if has_neg:
                cnt = int((df[c] < 0).sum())
                issues.append((c, "negative_values", cnt))
    return issues

# ----- Execução das checagens -----
report = []

# 1) Presença de colunas
req = check_required_columns(df_raw, REQUIRED_COLS)
report.append(("required_columns", "ok" if req["ok"] else "fail", req["missing_required_cols"]))

# 2) Tipos
dtype_errs = check_dtypes(df_raw, EXPECTED_DTYPES)
report.append(("dtypes", "ok" if not dtype_errs else "warn", dtype_errs))

# 3) Nulos nas chaves (veiculo_id, data)
nulls_key = check_nulls(df_raw, ["veiculo_id","data"], threshold=0.0)
report.append(("nulls_key", "ok" if not nulls_key else "fail", nulls_key))

# 4) Nulos em outras colunas
others = [c for c in REQUIRED_COLS if c not in ("veiculo_id","data")]
nulls_other = check_nulls(df_raw, others, threshold=0.0)
report.append(("nulls_other", "ok" if not nulls_other else "warn", nulls_other))

# 5) Duplicatas na chave (veiculo_id, data)
dups = check_duplicates(df_raw)
report.append(("duplicates_key", "ok" if dups == 0 else "warn", dups))

# 6) Flags binárias 0/1
flags = check_flags_binary(df_raw, ("flag_reset_odometro","fimdesemana"))
report.append(("flags_binary", "ok" if not flags else "warn", flags))

# 7) Não-negatividade (aviso)
nonneg_cols = ["km_dia_por_soma","km_dia_por_odometro","atividade_total_h","odo_ini_dia","odo_fim_dia"]
neg_issues = check_nonnegative(df_raw, nonneg_cols)
report.append(("nonnegative", "ok" if not neg_issues else "warn", neg_issues))

schema_report = pd.DataFrame(report, columns=["check","status","details"])
schema_report

,check,status,details
0,required_columns,ok,[]
1,dtypes,ok,[]
2,nulls_key,ok,[]
3,nulls_other,warn,"[(odo_ini_dia, null_rate, 0.8532242562656005),..."
4,duplicates_key,ok,0
5,flags_binary,ok,[]
6,nonnegative,ok,[]


Esse resultado indica que aproximadamente 71,6% das linhas não possuem valores preenchidos nas colunas odo_ini_dia e odo_fim_dia.

## 3. Saneamento do odômetro e seleção de `km_dia_final`

Nesta etapa, garantimos que os valores de quilometragem diária sejam **confiáveis, coerentes e consistentes**, mesmo quando os dados vêm de múltiplas fontes ou possuem ruídos operacionais. Para isso, aplicamos duas ações principais:

---

### ✅ **(A) Saneamento da variação do odômetro**

Primeiro calculamos a diferença diária:


Delta km = odo\_fim\_dia - odo\_ini\_dia


Em seguida aplicamos regras para evitar valores absurdos ou impossíveis:

- Se `Δ km < 0` → marcamos como inválido (**odômetro não pode regredir**)
- Se houver `flag_reset_odometro = 1` → forçamos o reset, ignorando a variação nesse dia
- Dias inválidos ficam temporariamente como `NaN` para tratamento seguro

Essas situações acontecem devido a:

- erro humano no registro
- troca de dispositivo
- reset proposital no painel
- falha de leitura do telemático

---

### ✅ **(B) Seleção da melhor fonte para o `km_dia_final`**

Como o sistema possui **duas formas de medir o KM diário**:

| Fonte | Campo | Prós | Contras |
|---------|---------|---------|---------|
| Diferença do odômetro | `delta_odo` | geralmente mais confiável | falha em caso de reset |
| Soma das viagens | `km_dia_por_soma` | útil como fallback | pode superestimar se houver ruído nas viagens |

Adotamos a regra:


km_{dia\_final} =
\begin{cases}
\Delta km, & \text{se odômetro for válido} \\\\
km\_dia\_por\_soma, & \text{caso contrário}
\end{cases}


E, por fim:

- valores negativos → **ajustados para 0**
- valores nulos → **preenchidos com 0**
- usamos `0` também para dias em que o veículo não rodou

---

### **Conclusão**

Com essa etapa, garantimos que:

- a quilometragem jamais diminui no tempo (consistência física)
- resets e ruídos não contaminam a série
- o `km_dia_final` é sempre **coeso, usável e comparável** entre veículos e dias

Essa coluna é a que será usada nas próximas etapas:  
**limpeza de outliers → features → forecasting → previsão de manutenção.**

---


In [ ]:
# ============================================================
# 3) Saneamento do odômetro e seleção do km_dia_final
# Requer: df_raw dos blocos anteriores (com 'data' datetime e 'data_br' opcional)
# ============================================================

# --- 1) Detecta a coluna de veículo ---
VEH_COL = None
for cand in ["veiculo_id","id_veiculo","veiculoid","idveiculo","vehicle_id","id_vehicle","id"]:
    if cand in df_raw.columns:
        VEH_COL = cand
        break
if VEH_COL is None:
    raise KeyError("Não encontrei a coluna do veículo (ex.: 'veiculo_id' / 'id_veiculo').")

# --- 2) Checa campos mínimos ---
MUST_HAVE = ["data", VEH_COL, "odo_ini_dia", "odo_fim_dia", "km_dia_por_soma"]
missing = [c for c in MUST_HAVE if c not in df_raw.columns]
if missing:
    raise KeyError(f"Colunas obrigatórias ausentes para o Bloco 4: {missing}")

# --- 3) Função por veículo (reinsere VEH_COL quando include_groups=False) ---
def _sane_odometer_and_select(g: pd.DataFrame, gb_key=None) -> pd.DataFrame:
    g = g.sort_values("data").copy()

    # (A) Repor a coluna do veículo se não veio no apply (pandas >= 2.2 com include_groups=False)
    if (VEH_COL not in g.columns) or (g[VEH_COL].isna().all() if VEH_COL in g.columns else True):
        if gb_key is None:
            gb_key = g[VEH_COL].iloc[0] if VEH_COL in g.columns else "veh_unknown"
        g[VEH_COL] = str(gb_key)

    # (B) Delta de odômetro bruto
    fim = pd.to_numeric(g.get("odo_fim_dia"), errors="coerce")
    ini = pd.to_numeric(g.get("odo_ini_dia"), errors="coerce")
    g["delta_odo"] = fim - ini

    # (C) Resets e negativos => delta inválido
    if "flag_reset_odometro" in g.columns:
        reset_mask = pd.to_numeric(g["flag_reset_odometro"], errors="coerce").fillna(0).astype("Int64") == 1
        g.loc[reset_mask, "delta_odo"] = np.nan
    g.loc[g["delta_odo"] < 0, "delta_odo"] = np.nan

    # (D) Fonte preferida do odômetro (km_dia_por_odometro se existir; senão delta_odo)
    if "km_dia_por_odometro" in g.columns:
        odo_pref = pd.to_numeric(g["km_dia_por_odometro"], errors="coerce")
        odo_pref = odo_pref.where(odo_pref.notna(), g["delta_odo"])
    else:
        odo_pref = g["delta_odo"]

    # (E) Fallback por soma de viagens
    soma = pd.to_numeric(g.get("km_dia_por_soma"), errors="coerce")

    # (F) Seleção final: odômetro > soma > zero
    km_sel = odo_pref.where(odo_pref.notna(), soma).fillna(0.0).clip(lower=0)
    g["km_dia_final"] = km_sel.astype("float64")

    # (G) Fonte de auditoria
    g["km_fonte"] = pd.Series(
        np.where(odo_pref.notna(), "odometro", np.where(soma.notna(), "soma", "zero")),
        index=g.index, dtype="string"
    )

    # (H) Horas do dia (não-negativas)
    if "atividade_total_h" in g.columns:
        h = pd.to_numeric(g["atividade_total_h"], errors="coerce").fillna(0.0).clip(lower=0)
        g["h_dia_final"] = h.astype("float64")
    else:
        g["h_dia_final"] = 0.0

    # (I) Flags auxiliares (diagnóstico)
    g["delta_odo_negativo_flag"] = (fim - ini) < 0
    g["delta_odo_nan_flag"] = g["delta_odo"].isna()

    return g

# --- 4) Execução do groupby.apply preservando VEH_COL (compatível com pandas) ---
gb = df_raw.groupby([VEH_COL], group_keys=False)
try:
    # pandas >= 2.2
    df4 = gb.apply(lambda g: _sane_odometer_and_select(g, gb_key=g.name), include_groups=False)
except TypeError:
    # pandas mais antigos (sem include_groups)
    df4 = gb.apply(lambda g: _sane_odometer_and_select(g, gb_key=g[VEH_COL].iloc[0]))

# --- 5) Auditoria e preview ---
total_linhas = len(df4)
por_fonte = df4["km_fonte"].value_counts(dropna=False, normalize=True).rename("proporcao").to_frame()
por_fonte["linhas"] = (por_fonte["proporcao"] * total_linhas).round().astype(int)

negativos_brutos = int(df4["delta_odo_negativo_flag"].sum())
nan_delta = int(df4["delta_odo_nan_flag"].sum())

print("=== Auditoria do Bloco 4 ===")
print(f"Linhas totais: {total_linhas:,}")
print("\nUso das fontes para km_dia_final:")
print(por_fonte)
print(f"\nDeltas negativos (antes de saneamento): {negativos_brutos:,}")
print(f"Deltas NaN (reset/sem info): {nan_delta:,}")

# Coluna 'data_br' para exibição (se ainda não existir)
if "data_br" not in df4.columns and "data" in df4.columns:
    df4["data_br"] = df4["data"].dt.strftime("%d/%m/%Y")

# Preview
cols_preview = [c for c in [VEH_COL,"data_br","km_dia_final","km_fonte","h_dia_final",
                            "km_dia_por_soma","km_dia_por_odometro",
                            "odo_ini_dia","odo_fim_dia","flag_reset_odometro"]
                if c in df4.columns]
df4[cols_preview].head(10)


=== Auditoria do Bloco 4 ===
Linhas totais: 22,124,296

Uso das fontes para km_dia_final:
          proporcao    linhas
km_fonte                     
odometro        1.0  22124296

Deltas negativos (antes de saneamento): 1,452
Deltas NaN (reset/sem info): 18,885,884


,veiculo_id,data_br,km_dia_final,km_fonte,h_dia_final,km_dia_por_soma,km_dia_por_odometro,odo_ini_dia,odo_fim_dia,flag_reset_odometro
0,1,01/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
1,1,02/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
2,1,03/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
3,1,04/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
4,1,05/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
5,1,06/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
6,1,07/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
7,1,08/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
8,1,09/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0
9,1,10/01/2022,0.0,odometro,0.0,0.0,0.0,<NA>,<NA>,0


## 4. Persistência


In [ ]:
# caminhos de saída
root = Path.cwd()
if not (root / "data").exists() and (root.parent / "data").exists():
    # se estiver rodando em notebooks/, sobe um nível
    root = root.parent

out_dir = root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = out_dir / "telemetria_moviasall.parquet"
csv_path     = out_dir / "telemetria_moviasall.csv"

# --- Salva Parquet com schema atual (data como datetime) ---
df4.to_parquet(parquet_path, index=False)

# --- Salva CSV com data em formato BR (dd/mm/aaaa) ---
df4_csv = df4.copy()
if "data" in df4_csv.columns:
    df4_csv["data"] = df4_csv["data"].dt.strftime("%d/%m/%Y")

df4_csv.to_csv(csv_path, index=False, encoding="utf-8")

print("Arquivos salvos:")
print(" - Parquet:", parquet_path)
print(" - CSV    :", csv_path)

Arquivos salvos:
 - Parquet: C:\Users\baj00\movias_forecast\data\processed\telemetria_moviasall.parquet
 - CSV    : C:\Users\baj00\movias_forecast\data\processed\telemetria_moviasall.csv



## 5. Outliers — Estratégia Híbrida (Hard Cap + IQR + Z-score opcional)
O objetivo deste bloco é remover valores extremos nas variáveis **`km_dia_final`** e **`h_dia_final`**, antes que esses dados sigam para modelagem, dashboards ou análise de manutenção. A ideia é garantir uma série temporal **realista, consistente e estável**, eliminando ruídos que podem distorcer qualquer tipo de previsão ou métrica.

---

####  1. Hard Cap (Limite Físico)

Aplicamos limites máximos possíveis no mundo real:

| Variável         | Limite Máximo | Justificativa |
|------------------|---------------|---------------|
| `km_dia_final`   | **2400 km/dia** | Mesmo em uso contínuo, é fisicamente impossível exceder esse valor |
| `h_dia_final`    | **24 h/dia**   | Um dia não pode ter mais do que 24 horas |

> Esses limites funcionam como **primeiro filtro**, corrigindo erros graves de telemetria, duplicações, falhas de comunicação ou inconsistências do sensor.

---

####  2. IQR (Interquartile Range) — por veículo

Após o Hard Cap, aplicamos o método IQR individualmente para cada veículo:

\[
\text{Limite Inferior} = Q1 - 1.5 \cdot IQR
\]
\[
\text{Limite Superior} = Q3 + 1.5 \cdot IQR
\]

Isso remove valores fora do padrão de uso daquele veículo, respeitando seu comportamento operacional e evitando que **perfis diferentes contaminem uns aos outros**.

> O IQR é um método **estatístico robusto**, bom contra outliers e menos sensível a valores extremos do que média e desvio padrão.

---

####  3. Z-score (opcional, somente com histórico ≥ 90 dias)

Para veículos com bastante histórico, aplicamos um terceiro filtro:

\[
Z = \frac{x - \mu}{\sigma}
\]

Valores com:

\[
|Z| > 4
\]

são suavizados para limites aceitáveis.  
Veículos com séries curtas **não aplicam Z-score**, evitando distorções indevidas.

---

###  Resultado das Colunas

Após o tratamento, geramos:

| Coluna            | Descrição |
|-------------------|-----------|
| `km_dia_clean`    | quilometragem diária tratada |
| `h_dia_clean`     | horas tratadas |
| `km_before` / `h_before` | valores originais antes do filtro |
| `km_changed` / `h_changed` | flags indicando se houve modificação |

Essas colunas **permitem auditoria total** e facilitam rastreabilidade.

---

###  Resumo da Estratégia

| Camada | Objetivo | Benefício |
|---------|----------|-----------|
| **Hard Cap** | respeitar a física | impede absurdos imediatamente |
| **IQR por veículo** | remover exceções estatísticas | respeita o perfil de uso individual |
| **Z-score (opcional)** | suavizar séries longas | reduz distorções residuais |

---

###  Resultado Final

Após esse bloco, garantimos um dataset:

- **realista** (sem valores fisicamente impossíveis)
- **estatisticamente coerente**
- **audível e confiável**
- **pronto para modelagem e predições**

As colunas finais a serem utilizadas nos próximos blocos são:

- `km_dia_clean`  
- `h_dia_clean`

Essas serão nossa base para **feature engineering**, modelos preditivos e cálculos de disponibilidade.

---

In [ ]:
# --- Preamble robusto ---
try:
    get_ipython().run_line_magic('xmode', 'Plain')
except Exception:
    pass  # segue em frente se não estiver num shell IPython

import os, gc, warnings, sys
os.environ["IPYTHON_SIMPLE_PROMPT"] = "1"
warnings.filterwarnings("ignore")
# Opcional: encurtar ainda mais o traceback do Python
# sys.tracebacklimit = 0  # (descomente se quiser esconder tracebacks)

gc.collect()

Exception reporting mode: Plain


17080

In [ ]:
# ============================================================
# 5) Outliers em Streaming (robusto para arquivos grandes)
#   - Lê CSV gigante em chunks
#   - Particiona por veículo em /data/tmp/veh_chunks
#   - Processa veículo a veículo em "shards" (baixa RAM, com checkpoint)
#   - Salva resultado consolidado em data/processed/telemetria_moviasall_outliers.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import sys, os, gc

# ------------------ Configuração ------------------
root = Path.cwd()
if not (root / "data").exists() and (root.parent / "data").exists():
    root = root.parent

in_dir       = root / "data" / "processed"
csv_in       = in_dir / "telemetria_moviasall.csv"   # entrada GIGANTE
tmp_dir      = root / "data" / "tmp" / "veh_chunks"  # temporários por veículo
out_dir      = root / "data" / "processed"
out_csv_path = out_dir / "telemetria_moviasall_outliers.csv"

CHUNKSIZE    = 100_000   # ajuste conforme sua máquina (100_000 / 500_000 / 1_000_000)
ENCODING     = "utf-8"
DAYFIRST     = True

# Lote de veículos por shard (ajuste se RAM for limitada)
SHARD_SIZE   = 500

# Lista mínima de colunas que precisamos neste bloco
CAND_VEH_COLS = ["veiculo_id","id_veiculo","veiculoid","idveiculo","vehicle_id","id_vehicle","id"]
NEEDED_COLS   = [
    "data",
    "km_dia_final","h_dia_final",
    "km_dia_por_soma","km_dia_por_odometro",
    "odo_ini_dia","odo_fim_dia",
    "flag_reset_odometro",
    "atividade_total_h",  # usada como fallback para h_dia_final
    "data_br",            # se já existir
]

# Hard caps
HARD_CAP_KM = 2400.0
HARD_CAP_H  = 24.0

# Pastas
tmp_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
DONE_DIR = tmp_dir / "_done"
DONE_DIR.mkdir(parents=True, exist_ok=True)


# -------- Funções utilitárias --------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.rename(columns=lambda c: c.replace("\ufeff", "") if isinstance(c, str) else c, inplace=True)
    df.columns = (df.columns
                  .str.strip()
                  .str.replace(r"\s+", "_", regex=True)
                  .str.lower())
    return df

def detect_vehicle_col(cols) -> str | None:
    for c in CAND_VEH_COLS:
        if c in cols:
            return c
    return None

def cap_iqr(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    q1 = s.quantile(0.25); q3 = s.quantile(0.75)
    if pd.isna(q1) or pd.isna(q3):
        return s.clip(lower=0)
    iqr = q3 - q1
    if iqr == 0:
        return s.clip(lower=0)
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    return s.clip(lower=max(0, lo), upper=hi)

def apply_outliers_one_vehicle(g: pd.DataFrame, veh_col: str) -> pd.DataFrame:
    """Aplica a estratégia híbrida para UM veículo (DataFrame já filtrado para um ID)."""
    g = g.sort_values("data").copy()

    # km_before
    if "km_dia_final" in g.columns:
        km_before = pd.to_numeric(g["km_dia_final"], errors="coerce")
    else:
        odo  = pd.to_numeric(g.get("km_dia_por_odometro"), errors="coerce")
        soma = pd.to_numeric(g.get("km_dia_por_soma"),   errors="coerce")
        km_before = odo.where(odo.notna(), soma).fillna(0.0)

    # h_before
    if "h_dia_final" in g.columns:
        h_before = pd.to_numeric(g["h_dia_final"], errors="coerce")
    else:
        h_before = pd.to_numeric(g.get("atividade_total_h"), errors="coerce").fillna(0.0)

    # Hard cap + IQR
    km_cap = km_before.fillna(0.0).clip(upper=HARD_CAP_KM)
    h_cap  = h_before.fillna(0.0).clip(upper=HARD_CAP_H)

    km_iqr = cap_iqr(km_cap)
    h_iqr  = cap_iqr(h_cap)

    # Z-score opcional (se série longa)
    if len(g) >= 90:
        mu_km = float(km_iqr.mean()); sd_km = float(km_iqr.std(ddof=0)) or 1.0
        z_km  = (km_iqr - mu_km) / sd_km
        km_final = km_iqr.mask(z_km.abs() > 4, mu_km + 4*np.sign(z_km)*sd_km).clip(lower=0)

        mu_h = float(h_iqr.mean()); sd_h = float(h_iqr.std(ddof=0)) or 1.0
        z_h  = (h_iqr - mu_h) / sd_h
        h_final = h_iqr.mask(z_h.abs() > 4, mu_h + 4*np.sign(z_h)*sd_h).clip(lower=0)
    else:
        km_final = km_iqr.clip(lower=0)
        h_final  = h_iqr.clip(lower=0)

    g["km_dia_clean"] = km_final.astype("float32")
    g["h_dia_clean"]  = h_final.astype("float32")

    # data_br p/ export
    if "data_br" not in g.columns and "data" in g.columns:
        g["data_br"] = g["data"].dt.strftime("%d/%m/%Y")

    # Garante tipo do id
    g[veh_col] = g[veh_col].astype("string")

    # Mantém apenas o essencial antes de retornar (deixa mais leve)
    keep_cols = [veh_col, "data", "data_br", "km_dia_clean", "h_dia_clean"]
    keep_cols = [c for c in keep_cols if c in g.columns]
    return g[keep_cols].copy()


def veh_done_mark(vid: str):
    (DONE_DIR / f"{vid}.done").write_text("ok", encoding="utf-8")

def veh_is_done(vid: str) -> bool:
    return (DONE_DIR / f"{vid}.done").exists()


# ------------------ Passo 1: Particionar CSV gigante por veículo ------------------
print(">> Passo 1/3: Particionando CSV em arquivos por veículo (streaming/chunks)")
tmp_dir.mkdir(parents=True, exist_ok=True)

# Descobre cabeçalho e coluna de veículo
head_cols = pd.read_csv(csv_in, nrows=0, encoding=ENCODING).columns
head_cols = pd.Index([c.replace("\ufeff","") if isinstance(c,str) else c for c in head_cols])
head_cols = head_cols.str.strip().str.replace(r"\s+","_",regex=True).str.lower()
veh_col = detect_vehicle_col(head_cols)
if veh_col is None:
    raise KeyError("Não encontrei a coluna do veículo no cabeçalho do CSV.")

# Ler em chunks e escrever por veículo (idempotente: reaproveita arquivos existentes)
rows_total = 0
chunk_iter = pd.read_csv(
    csv_in,
    chunksize=CHUNKSIZE,
    encoding=ENCODING,
    low_memory=True,      # mantém memória sob controle
    dtype=str,            # lê como string e converte por coluna só quando necessário
)

for i, ch in enumerate(chunk_iter, 1):
    ch = normalize_columns(ch)
    if veh_col not in ch.columns:
        raise KeyError(f"O chunk {i} veio sem a coluna de veículo '{veh_col}'. Verifique o arquivo.")

    # Seleciona colunas úteis (as que existirem)
    keep = [veh_col, "data"] + [c for c in NEEDED_COLS if c in ch.columns]
    keep = list(dict.fromkeys([c for c in keep if c in ch.columns]))  # ordem + únicos
    ch = ch[keep].copy()

    # Converte data
    ch["data"] = pd.to_datetime(ch["data"], dayfirst=DAYFIRST, errors="coerce")

    # Split por veículo e apenda
    for vid, g in ch.groupby(veh_col, sort=False):
        if pd.isna(vid):
            continue
        fpath = tmp_dir / f"{veh_col}={str(vid)}.csv"
        header = not fpath.exists()
        g.to_csv(fpath, mode="a", header=header, index=False, encoding=ENCODING)

    rows_total += len(ch)
    if i % 10 == 0:
        print(f"  - Chunk {i}: {len(ch):,} linhas (total {rows_total:,})")

print(">> Particionamento concluído.")


# ------------------ Passo 2: Processar veículo a veículo (shards) ------------------
print(">> Passo 2/3: Aplicando tratamento de outliers por veículo e consolidando saída")

# Prepara arquivo de saída (cabeçalho apenas uma vez)
wrote_header = out_csv_path.exists() and (out_csv_path.stat().st_size > 0)
count_total = 0
batch = []

def process_batch(batch_list):
    global wrote_header, count_total
    for vid_, fpath_ in batch_list:
        if veh_is_done(vid_):
            continue
        try:
            g = pd.read_csv(fpath_, encoding=ENCODING)
            g = normalize_columns(g)
            if "data" not in g.columns:
                continue
            g["data"] = pd.to_datetime(g["data"], errors="coerce")

            g_out = apply_outliers_one_vehicle(g, veh_col=veh_col)
            # grava
            g_out.sort_values(["data"], inplace=True)
            g_out.to_csv(out_csv_path, mode="a", header=(not wrote_header), index=False, encoding=ENCODING)
            wrote_header = True

            veh_done_mark(vid_)
            count_total += 1

        except MemoryError:
            print(f"[MEM] Sem memória ao processar veículo {vid_}. Progresso salvo até aqui.", file=sys.stderr)
            raise
        except Exception as e:
            # Log compacto para não inflar memória
            print(f"[AVISO] Falha no veículo {vid_}: {type(e).__name__}: {e}", file=sys.stderr)
        finally:
            # Libera RAM agressivamente
            try:
                del g_out
            except:  # noqa
                pass
            try:
                del g
            except:  # noqa
                pass
            gc.collect()

# Itere em fluxo (sem listar tudo na memória)
for fpath in tmp_dir.glob(f"{veh_col}=*.csv"):
    vid = fpath.stem.split("=", 1)[1]
    if veh_is_done(vid):
        continue
    batch.append((vid, fpath))

    if len(batch) >= SHARD_SIZE:
        process_batch(batch)
        batch.clear()
        gc.collect()
        if count_total % 500 == 0 and count_total > 0:
            print(f"  - Veículos processados até agora: {count_total:,}")

# Processa o restante
if batch:
    process_batch(batch)
    batch.clear()
    gc.collect()

print(f">> Consolidação concluída: {out_csv_path} (veículos processados: {count_total:,})")

# ------------------ Passo 3: (Opcional) Estatísticas rápidas ------------------
# Ex.: poderia ler incrementalmente o CSV final e produzir um resumo.
# Mantido vazio para evitar consumir RAM desnecessária aqui.
print(">> Passo 3/3: Finalizado.")


>> Passo 1/3: Particionando CSV em arquivos por veículo (streaming/chunks)
  - Chunk 10: 100,000 linhas (total 1,000,000)
  - Chunk 20: 100,000 linhas (total 2,000,000)
  - Chunk 30: 100,000 linhas (total 3,000,000)
  - Chunk 40: 100,000 linhas (total 4,000,000)
  - Chunk 50: 100,000 linhas (total 5,000,000)
  - Chunk 60: 100,000 linhas (total 6,000,000)
  - Chunk 70: 100,000 linhas (total 7,000,000)
  - Chunk 80: 100,000 linhas (total 8,000,000)
  - Chunk 90: 100,000 linhas (total 9,000,000)
  - Chunk 100: 100,000 linhas (total 10,000,000)
  - Chunk 110: 100,000 linhas (total 11,000,000)
  - Chunk 120: 100,000 linhas (total 12,000,000)
  - Chunk 130: 100,000 linhas (total 13,000,000)
  - Chunk 140: 100,000 linhas (total 14,000,000)
  - Chunk 150: 100,000 linhas (total 15,000,000)
  - Chunk 160: 100,000 linhas (total 16,000,000)
  - Chunk 170: 100,000 linhas (total 17,000,000)
  - Chunk 180: 100,000 linhas (total 18,000,000)
  - Chunk 190: 100,000 linhas (total 19,000,000)
  - Chunk 20

In [ ]:
# ------------------ Passo 3: (Opcional) Limpar temporários ------------------

shutil.rmtree(tmp_dir, ignore_errors=True)
print(">> Temporários removidos:", tmp_dir)

>> Temporários removidos: C:\Users\baj00\movias_forecast\data\tmp\veh_chunks



## 6. Features derivadas (cumulativos, médias móveis, sazonalidade, inatividade)

Este bloco enriquece a base **após o tratamento de outliers (Bloco 5)** com *features* de tendência, sazonalidade e inatividade, **calculadas por veículo** (PV), preservando todos os dados.

**Entradas (do Bloco 5):**
- `km_dia_clean` — km diário já tratado
- `h_dia_clean` — horas diárias já tratadas
- `data` — datetime diário contínuo
- `veiculo_id` (ou alias detectado)

**Janelas usadas (dias):** 7, 14, 21, 28  
Motivação: semanal (7), quinzena (14), ciclo operacional (21), mensal aproximado (28).

#### Features geradas

**Tendência (rolling por veículo):**
- Médias móveis de km: `mm_km_7`, `mm_km_14`, `mm_km_21`, `mm_km_28`
- Somas móveis de km: `sm_km_7`, `sm_km_14`, `sm_km_21`, `sm_km_28` *(opcional, útil para janelas acumuladas)*
- Médias móveis de horas: `mm_h_7`, `mm_h_14`, `mm_h_21`, `mm_h_28`

**Sazonalidade:**
- `weekday` (0=Seg … 6=Dom)
- `fimdesemana` (recalculado por calendário, 1=Sab/Dom)
- `mes` (1–12)
- `sin_sem`, `cos_sem` (codificação cíclica semanal)

**Inatividade:**
- `flag_parado` = 1 se `km_dia_clean == 0`, senão 0
- `dias_parado_seq` = dias consecutivos parado (reset ao rodar)
- `% inatividade (7/14d)`: `pct_parado_7d`, `pct_parado_14d` = média móvel de `flag_parado`

> Observação: todas as features são calculadas **por veículo** e com `min_periods=1`, para que os primeiros dias não fiquem vazios.

**Saída:** `data/processed/telemetria_moviaas2025_features.*`  
(Parquet se engine disponível; sempre CSV com `data` no formato BR)

In [ ]:
# ============================================================
# 6) Feature Engineering (PV, streaming por veículo, low-memory, retomável)
#   - Entrada: data/processed/telemetria_moviasall_outliers.csv
#   - Saída  : data/processed/telemetria_moviasall_features.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import sys, os, gc

# ------------------ Config ------------------
root = Path.cwd()
if not (root / "data").exists() and (root.parent / "data").exists():
    root = root.parent

proc_dir      = root / "data" / "processed"
tmp_dir       = root / "data" / "tmp" / "veh_chunks_b6"
out_csv_path  = proc_dir / "telemetria_moviasall_features.csv"
out_parq_dir  = proc_dir / "telemetria_moviasall_features_parquet"   # opcional

in_csv  = proc_dir / "telemetria_moviasall_outliers.csv"
in_parq = proc_dir / "telemetria_moviasall_outliers.parquet"

CHUNKSIZE   = 100_000
ENCODING    = "utf-8"
DAYFIRST    = True
SHARD_SIZE  = 500

WINDOWS = [7, 14, 21, 28]

tmp_dir.mkdir(parents=True, exist_ok=True)
DONE_DIR = tmp_dir / "_done"
DONE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------ Utils ------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.rename(columns=lambda c: c.replace("\ufeff", "") if isinstance(c, str) else c, inplace=True)
    df.columns = (df.columns
                  .str.strip()
                  .str.replace(r"\s+", "_", regex=True)
                  .str.lower())
    return df

def detect_vehicle_col(cols) -> str | None:
    for c in ["veiculo_id","id_veiculo","veiculoid","idveiculo","vehicle_id","id_vehicle","id"]:
        if c in cols:
            return c
    return None

def veh_done_mark(vid: str):
    (DONE_DIR / f"{vid}.done").write_text("ok", encoding="utf-8")

def veh_is_done(vid: str) -> bool:
    return (DONE_DIR / f"{vid}.done").exists()

def parse_date_with_fallback(df: pd.DataFrame) -> pd.Series:
    """
    Prioriza 'data' (dd/mm/aaaa). Se vier tudo NaT, tenta 'data_original_str' (ISO) e por fim 'data_br'.
    """
    dt = pd.to_datetime(df.get("data"), dayfirst=DAYFIRST, errors="coerce")
    if dt.notna().any():
        return dt
    if "data_original_str" in df.columns:
        dt2 = pd.to_datetime(df["data_original_str"], errors="coerce")
        if dt2.notna().any():
            return dt2
    if "data_br" in df.columns:
        dt3 = pd.to_datetime(df["data_br"], dayfirst=True, errors="coerce")
        return dt3
    return dt  # tudo NaT mesmo

def rolling_features_vehicle(g: pd.DataFrame, veh_col: str) -> pd.DataFrame:
    """Calcula features rolling/sazonais/inatividade para UM veículo, evitando NaN->int."""
    # Limpa e ordena
    g = g.copy()
    g.replace([np.inf, -np.inf], np.nan, inplace=True)
    g["data"] = parse_date_with_fallback(g)
    g = g[g["data"].notna()].sort_values("data")
    if g.empty:
        # Retorna DF vazio com as colunas esperadas para não quebrar o append
        base_cols = [veh_col, "data", "data_br", "km_dia_clean", "h_dia_clean",
                     "weekday","fimdesemana","mes","sin_sem","cos_sem",
                     "mm_km_7","sm_km_7","mm_h_7","mm_km_14","sm_km_14","mm_h_14",
                     "mm_km_21","sm_km_21","mm_h_21","mm_km_28","sm_km_28","mm_h_28",
                     "flag_parado","dias_parado_seq","pct_parado_7d","pct_parado_14d"]
        return pd.DataFrame(columns=[c for c in base_cols if c not in g.columns])

    # Bases numéricas
    km = pd.to_numeric(g.get("km_dia_clean"), errors="coerce").fillna(0).astype("float32")
    hh = pd.to_numeric(g.get("h_dia_clean"),  errors="coerce").fillna(0).astype("float32")

    # Rolling
    for w in WINDOWS:
        g[f"mm_km_{w}"] = km.rolling(window=w, min_periods=1).mean().astype("float32")
        g[f"sm_km_{w}"] = km.rolling(window=w, min_periods=1).sum().astype("float32")
        g[f"mm_h_{w}"]  = hh.rolling(window=w, min_periods=1).mean().astype("float32")

    # Se existir fimdesemana vindo do Bloco 5, removemos para recriar
    g.drop(columns=["fimdesemana"], errors="ignore", inplace=True)

    # Sazonalidade (sem NaN em ints)
    wd = g["data"].dt.weekday
    wd = wd.where(g["data"].notna(), -1).fillna(-1).astype("int16")
    g["weekday"] = wd
    g["fimdesemana"] = (wd >= 5).astype("int8")

    mes = g["data"].dt.month
    mes = mes.where(g["data"].notna(), 0).fillna(0).astype("int8")
    g["mes"] = mes

    wd_pos = np.where(wd.values >= 0, wd.values, 0).astype("float32")
    ang = (2 * np.pi * (wd_pos / np.float32(7.0))).astype("float32")
    g["sin_sem"] = np.sin(ang).astype("float32")
    g["cos_sem"] = np.cos(ang).astype("float32")

    # Inatividade
    flag_parado = (km == 0).astype("int8")
    g["flag_parado"] = flag_parado

    start_seq = ((flag_parado == 1) & (flag_parado.shift(1).fillna(0).astype("int8") == 0)).astype("int8")
    bloco_id  = start_seq.cumsum()
    seq = g.groupby(bloco_id, sort=False)["flag_parado"].cumcount() + 1
    g["dias_parado_seq"] = np.where(flag_parado == 1, seq, 0).astype("int16")

    g["pct_parado_7d"]  = flag_parado.rolling(window=7,  min_periods=1).mean().astype("float32")
    g["pct_parado_14d"] = flag_parado.rolling(window=14, min_periods=1).mean().astype("float32")

    # data_br
    if "data_br" not in g.columns:
        g["data_br"] = g["data"].dt.strftime("%d/%m/%Y")

    # sobrescreve bases com tipos finais
    g["km_dia_clean"] = km
    g["h_dia_clean"]  = hh

    # posiciona id primeiro
    g[veh_col] = g[veh_col].astype("string")
    cols = [veh_col] + [c for c in g.columns if c != veh_col]
    return g[cols]

# ------------------ Seleção de entrada e particionamento ------------------
in_path, is_csv = None, None
if in_csv.exists():
    in_path, is_csv = in_csv, True
elif in_parq.exists():
    in_path, is_csv = in_parq, False
else:
    raise FileNotFoundError("Nenhum arquivo de entrada encontrado: telemetria_moviasall_outliers.(csv|parquet)")

veh_col = None
if is_csv:
    print(f"[INFO] Lendo em streaming de: {in_path}")
    head_cols = pd.read_csv(in_path, nrows=0, encoding=ENCODING).columns
    head_cols = pd.Index([c.replace("\ufeff","") if isinstance(c,str) else c for c in head_cols])
    head_cols = head_cols.str.strip().str.replace(r"\s+","_",regex=True).str.lower()
    veh_col = detect_vehicle_col(head_cols)
    if veh_col is None:
        raise KeyError("Não encontrei a coluna de veículo no cabeçalho do CSV de entrada.")

    rows_total = 0
    for i, ch in enumerate(pd.read_csv(in_path,
                                       chunksize=CHUNKSIZE,
                                       encoding=ENCODING,
                                       low_memory=True,
                                       dtype=str), 1):
        ch = normalize_columns(ch)
        keep = [veh_col, "data", "data_original_str", "data_br", "km_dia_clean", "h_dia_clean", "fimdesemana"]
        keep = [c for c in keep if c in ch.columns]
        ch = ch[keep].copy()

        # parse de data só no passo 2; aqui só particiona
        for vid, g in ch.groupby(veh_col, sort=False):
            if pd.isna(vid):
                continue
            fpath = tmp_dir / f"{veh_col}={str(vid)}.csv"
            header = not fpath.exists()
            g.to_csv(fpath, mode="a", header=header, index=False, encoding=ENCODING)

        rows_total += len(ch)
        if i % 50 == 0:
            print(f"  - Chunk {i}: {len(ch):,} linhas (total {rows_total:,})")
        del ch
        gc.collect()
    print("[OK] Particionamento concluído.")
else:
    print(f"[AVISO] CSV não encontrado; tentando Parquet: {in_path}")
    df = pd.read_parquet(in_path)
    df = normalize_columns(df)
    veh_col = detect_vehicle_col(df.columns)
    if veh_col is None:
        raise KeyError("Não encontrei a coluna de veículo no Parquet de entrada.")
    for vid, g in df.groupby(veh_col, sort=False):
        if pd.isna(vid):
            continue
        g.to_csv(tmp_dir / f"{veh_col}={str(vid)}.csv", index=False, encoding=ENCODING)
    del df
    gc.collect()
    print("[OK] Particionamento (via Parquet) concluído.")

# ------------------ Processar shards com checkpoint ------------------
print("[INFO] Calculando features por veículo e consolidando saída…")

wrote_header = out_csv_path.exists() and (out_csv_path.stat().st_size > 0)
count_total = 0
batch = []

def process_batch(batch_list):
    global wrote_header, count_total
    for vid_, fpath_ in batch_list:
        if veh_is_done(vid_):
            continue
        try:
            g = pd.read_csv(fpath_, encoding=ENCODING)
            g = normalize_columns(g)

            # Blindagem geral antes das features
            g.replace([np.inf, -np.inf], np.nan, inplace=True)
            # km/h base garantidas como numéricas
            for base_col in ("km_dia_clean","h_dia_clean"):
                if base_col not in g.columns:
                    g[base_col] = 0.0
                g[base_col] = pd.to_numeric(g[base_col], errors="coerce").fillna(0.0)

            # Aplica features
            g_feat = rolling_features_vehicle(g, veh_col)
            if g_feat.empty:
                veh_done_mark(vid_)
                continue

            # Downcast extra (se algo ficou em float64)
            for c in g_feat.select_dtypes(include=["float64"]).columns:
                g_feat[c] = g_feat[c].astype("float32")

            g_feat.sort_values(["data"], inplace=True)
            g_feat.to_csv(out_csv_path, mode="a", header=(not wrote_header), index=False, encoding=ENCODING)
            wrote_header = True

            veh_done_mark(vid_)
            count_total += 1

        except MemoryError:
            print(f"[MEM] Sem memória no veículo {vid_}. Progresso salvo; execute novamente.", file=sys.stderr)
            raise
        except Exception as e:
            print(f"[AVISO] Falha no veículo {vid_}: {type(e).__name__}: {e}", file=sys.stderr)
        finally:
            try: del g_feat
            except: pass
            try: del g
            except: pass
            gc.collect()

for fpath in tmp_dir.glob(f"{veh_col}=*.csv"):
    vid = fpath.stem.split("=", 1)[1]
    if veh_is_done(vid):
        continue
    batch.append((vid, fpath))
    if len(batch) >= SHARD_SIZE:
        process_batch(batch)
        batch.clear()
        gc.collect()
        if count_total % 500 == 0 and count_total > 0:
            print(f"  - Veículos processados até agora: {count_total:,}")

if batch:
    process_batch(batch)
    batch.clear()
    gc.collect()

print(f"[OK] Saída CSV consolidada: {out_csv_path} (veículos processados: {count_total:,})")


[INFO] Lendo em streaming de: C:\Users\baj00\movias_forecast\data\processed\telemetria_moviasall_outliers.csv
  - Chunk 50: 100,000 linhas (total 5,000,000)
  - Chunk 100: 100,000 linhas (total 10,000,000)
  - Chunk 150: 100,000 linhas (total 15,000,000)
  - Chunk 200: 100,000 linhas (total 20,000,000)
  - Chunk 250: 100,000 linhas (total 25,000,000)
[OK] Particionamento concluído.
[INFO] Calculando features por veículo e consolidando saída…
  - Veículos processados até agora: 500
  - Veículos processados até agora: 1,000
  - Veículos processados até agora: 1,500
  - Veículos processados até agora: 2,000
  - Veículos processados até agora: 2,500
  - Veículos processados até agora: 3,000
  - Veículos processados até agora: 3,500
  - Veículos processados até agora: 4,000
  - Veículos processados até agora: 4,500
  - Veículos processados até agora: 5,000
  - Veículos processados até agora: 5,500
  - Veículos processados até agora: 6,000
  - Veículos processados até agora: 6,500
  - Veícul

In [ ]:
# ------------------ 4) (Opcional) Limpar temporários ------------------

shutil.rmtree(tmp_dir, ignore_errors=True)
print("[OK] Temporários removidos:", tmp_dir)



## 7. Próximos passos (para o Forecasting)
1. Prever km/horas futuras (SARIMA/ETS/LSTM).  
2. Derivar data-alvo (quando atinge limiar).  
3. Calcular janelas de confiança (quantis).  
4. Persistir saídas em `forecast_results`.
